## Parte 1 — Observabilidad

**Haz que tu agente deje spans —un registro por cada paso de trabajo— que se puedan consultar sin abrir el código,
con Langfuse (obligatorio). LangSmith o un JSONL propio pueden ir además, no en su lugar. No se piden capturas
de pantalla: se pide el archivo de donde salen las cifras, que se exporta de Langfuse.**


**1. Cuatro tipos de span, con estos campos como mínimo:**

| **Span** | **Qué registra** | **Cómo en Langfuse** |
|---|---|---|
| **`corrida`** | **La pregunta completa, su `status` y su `prompt_version`** | **La observation raíz (`@observe` en `.run()`)** |
| **`guardrail`** | **Cada validación y su decisión (`ok`, `redactada`, `bloqueada`)** | **`as_type="guardrail"`, con la decisión como `output`** |
| **`llm`** | **Cada llamada, con modelo, tokens de entrada y salida y latencia** | **Una `generation` (el cliente `from langfuse.openai import OpenAI` la registra sola)** |
| **`herramienta`** | **Cada uso, con nombre, tamaño del resultado, latencia y error** | **`as_type="tool"`** |

**2. Corrige los tres problemas de la 0.c:** **mide la duración en un `finally` (así existe el span aunque falle, con su error); que el cronómetro incluya los guardrails; y redacta los datos personales antes de escribir la traza.** **Con Langfuse: el span se cierra solo aunque el código falle; la raíz tiene que envolver los guardrails; y la redacción es la máscara `Langfuse(mask=tu_funcion)`, que debe recorrer también tuplas (así llegan los argumentos).**

**3. Costo:** **con la H200 es cero. Calcula también cuánto costaría con un modelo de pago, con el precio de su fila y su fecha:**

```bash
python resumir_trazas.py spans.jsonl --precio propietario_economico
```

**4. `traza_id`:** **`.run()` devuelve también el `traza_id` —con Langfuse, `get_client().get_current_trace_id()` dentro de la raíz—, para ir de una fila del CSV a sus spans.**

**El archivo:** **pon a cada corrida su sesión y etiqueta con `propagate_attributes(session_id="grupo-07-base", tags=["taller-04"])`, llama a `flush()` al terminar, y expórtala desde `Lab-04-LLMOps/`:**

```bash
python exportar_trazas.py --sesion grupo-07-base -o spans.jsonl
python resumir_trazas.py spans.jsonl --precio propietario_economico
```

**`exportar_trazas.py` escribe cada span con esta forma (es lo que lee `resumir_trazas.py`):**

```json
{"traza": "id-de-la-corrida", "span": "llm", "nombre": "modelo-o-herramienta", "duracion_ms": 912.4, "modelo": "…", "tokens_entrada": 1500, "tokens_salida": 93, "error": null, "prompt_version": "v1"}
```

**`span` es `corrida`, `llm`, `herramienta` o `guardrail`. Los tokens van solo en los `llm`; la `prompt_version`, en el span `corrida`.**

**Opciones B y C: un span por agente.** **Cada rol del solver (B) o cada nodo del grafo (C) deja su observation con `as_type="agent"` bajo la raíz de la corrida, y las llamadas al modelo y a las herramientas quedan debajo del agente que las pidió. En LangGraph basta con pasar el manejador de Langfuse al grafo, dentro de la función raíz:**

```python
from langfuse.langchain import CallbackHandler

resultado = grafo.invoke(estado, config={"callbacks": [CallbackHandler()]})
```

**La consulta al RAG va como `as_type="retriever"`, con los fragmentos como salida. En tu tabla, reparte el tiempo y los tokens por agente y di cuál gasta más.**

### Entregables:

Los spans de la corrida del golden set, exportados de LangFuse corresponden a 355 spans de 5 trazas, una por tarea y se encuentra en la siguiente ubicación dentro de la carpeta `Lab-04-LLMOps/results_parte1`:


["spans.jsonl"](/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/spans.jsonl)


Las corridas que representan una fila por corrida fueron exportadas en un archivo .csv, y se especifica duración, tokens de entrada y de salida, llamadas al modelo y spans con error, se encuentran dentro de `Lab-04-LLMOps/results_parte1` en la siguiente ubicación:




["corridas.csv"](/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/tablas/corridas.csv)



El total de tokens de entrada y salida y la latencia p50 y p95 por corrida se encuentra dentro de  `Lab-04-LLMOps/results_parte1`      en la siguiente ubicación: 



["resumen_de_corridas"](/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/tablas/resumen_de_corridas.csv)


A continuación los resultados:

| corridas | tokens_entrada | tokens_salida | latencia_p50_s | latencia_p95_s | latencia_max_s |
|---------:|---------------:|--------------:|---------------:|---------------:|---------------:|
| 5 | 306169 | 870186 | 468.313 | 1152.551 | 1272.072 |


El costo por fila y flecha se encuentra en:

["resumen_trazas.txt"](/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/resumen_trazas.txt)


Otros archivos generados tras la corrida fueron:

Para determinar los tokens y tiempo por agente, y cuál gasta más: 

["tiempo_por_tipo_de_span.csv"](
/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/tablas/tiempo_por_tipo_de_span.csv)



Tiempo por agente en cada corrida: 

["tokens_y_tiempo_por_agente.csv"](
/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/tablas/tokens_y_tiempo_por_agente.csv)


Evidencia de la corrida de cada tarea A,B,C,S1,S2 se encuentra en: 

["corridas/base-r1"](
/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/corridas/base-r1)



El costo del índice en frío:  

["indice.json"](
/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/indice_cache/indice.json)

["traza_notas.jsonl"](
/home/dludena/Taller4_IA_Generativa_Agentes/Lab-04-LLMOps/results_parte1/indice_cache/traza_notas.jsonl)



La siguiente tabla muestra el resumen de corridas, tokens, costo y latencia.


| Métrica | Valor |
|---|---:|
| Corridas (golden set, sesión `grupo-07-base`) | 5 |
| Tokens de entrada | 306,169 |
| Tokens de salida | 870,186 |
| Tokens totales | 1,176,355 |
| Costo real (H200, modelo local `zai-org/GLM-5.3-Flash`) | 0 USD |
| Costo hipotético con la fila `propietario_economico` (OpenAI `gpt-4o-mini`; 0.15 USD entrada / 0.60 USD salida por 1M tokens; verificada el 2026-08-26) | **0.568037 USD** |
| Latencia p50 por corrida | 468.31 s |
| Latencia p95 por corrida | 1,152.55 s |
| Latencia máxima | 1,272.07 s |

Se muestra en esta tabla el tiempo por tipo de span:



| Tipo de span | Spans | Tiempo total (s) | Promedio (s) | Errores |
|---|---:|---:|---:|---:|
| `llm` | 85 | 3,035.66 | 35.71 | 0 |
| `corrida` | 5 | 2,721.26 | 544.25 | 1 |
| `agent` | 111 | 2,721.02 | 24.51 | 0 |
| `herramienta` | 24 | 20.07 | 0.84 | 0 |
| `guardrail` | 130 | 0.12 | 0.00 | 0 |

Los tiempos de corrida y agent no se suman con los demás, porque contienen a los spans hijos y se contaría el mismo segundo dos veces.
Los spans llm suman más (3,035.66 s) que el tiempo real de las corridas (2,721.26 s) porque las llamadas del indexador del GraphRAG corren en paralelo (concurrencia ≈ 3.5).
El único error es el span raíz corrida de la tarea S2, que falló por presupuesto agotado. resumen_trazas.txt dice errores: 0 porque no cuenta el span raíz.

Los tokens y tiempos por agente se reflejan en la siguiente tabla:

| Agente | Activaciones | Tiempo de pared (s) | Llamadas LLM | Tokens de entrada | Tokens de salida | Suma LLM (s) | % del tiempo de corridas | Tokens totales | Concurrencia LLM |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `programador` | 22 | 1977.581 | 26 | 107618 | 595710 | 1977.477 | 72.6717 | 703328 | 0.9999 |
| `critico` | 21 | 414.119 | 20 | 127976 | 128777 | 413.902 | 15.2179 | 256753 | 0.9995 |
| `indexador` | 5 | 135.657 | 28 | 11954 | 95804 | 479.711 | 4.9851 | 107758 | 3.5362 |
| `redactor` | 11 | 86.902 | 5 | 45320 | 25736 | 84.328 | 3.1935 | 71056 | 0.9704 |
| `planificador` | 6 | 80.273 | 6 | 13301 | 24159 | 80.237 | 2.9499 | 37460 | 0.9996 |
| `ejecutor` | 21 | 19.506 | 0 | 0 | 0 | 0.000 | 0.7168 | 0 | 0.0000 |
| `investigador` | 20 | 6.262 | 0 | 0 | 0 | 0.000 | 0.2301 | 0 | 0.0000 |
| `lector` | 5 | 0.722 | 0 | 0 | 0 | 0.000 | 0.0265 | 0 | 0.0000 |

**¿Dónde se va el tiempo? (el cuello de botella es el que suma más tiempo total, no el de mayor promedio)**

El cuello de botella es el agente programador, que usa el LLM para generar y corregir el código de cada tarea. Ocupa 1,977.6 s de pared, el 73 % del tiempo total de las corridas, con un promedio de ≈76 s por llamada. También es el que más tokens gasta, cerca del 60 % del total, y el 80 % de los suyos son de salida (razonamiento y código). Es el agente más lento en las cinco corridas (entre el 56 % y el 83 % de cada una). El crítico ocupa ≈15 % y el indexador ≈5 %, porque el índice de notas se reutiliza desde caché. Construir ese índice en frío tomó 138 s, el 60 % de la tarea A. Las herramientas y los guardrails suman menos del 1 % (20.2 s en total), así que su costo de observabilidad es despreciable.

Entre los spans hoja (llm, herramienta, guardrail), llm aporta 3,035.66 s de 3,055.85 s (99.3 %).